In [ ]:
import matplotlib.pyplot as plt
import torch
import os
import torch.nn as nn
import ipywidgets as widgets
import torch.nn.functional as F
from IPython.display import display
import math
import random
import numpy as np
from models import HGNAN

def set_all_seeds(seed=42):
    random.seed(seed) 
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(42)

In [2]:
feature_names = {
  0: 'hair',
  1: 'feathers', 
  2: 'eggs',  
  3: 'milk', 
  4: 'airborne', 
  5: 'aquatic', 
  6: 'predator', 
  7: 'toothed', 
  8: 'backbone',
  9: 'breathes',  
  10: 'venomous',
  11: 'fins', 
  12: 'legs', 
  13: 'tail', 
  14: 'domestic', 
  15: 'catsize'}

In [28]:
class FScorePlotter:
    def __init__(self, model, data, data_name, feature_names, class_names):
        self.model = model
        self.data = data
        self.data_name = data_name
        self.feature_names = feature_names
        self.class_names = class_names
        self.num_features = data.size(1)
        self.binary_indices = self.compute_binary_indices()
        self.last_fig = None
        
        self.dropdown = widgets.Dropdown(
            options=[(f"{class_id}: {name}", class_id) for class_id, name in self.class_names.items()],
            value=0,
            description='Class:',
            disabled=False,
        )
        self.filename_text = widgets.Text(
            value='',
            placeholder='Enter file name',
            description='File Name:',
            disabled=False,
        )
        self.save_button = widgets.Button(
            description="Save Plot",
            disabled=False,
            button_style='',
            tooltip='Click to save the current plot',
            icon='save'
        )
        self.save_button.on_click(self.save_plot)
    
    def compute_binary_indices(self):
        binary_indices = []
        num_features = self.data.size(1)
        for i in range(num_features):
            unique_vals = torch.unique(self.data[:, i])
            if unique_vals.numel() <= 2:
                binary_indices.append(i)
        return binary_indices
    
    def plot_f_scores(self, selected_class):
        plt.rcParams['font.family'] = 'STIXGeneral'
        plt.rcParams['mathtext.fontset'] = 'stix'
        plt.rcParams.update({
            'font.size': 20,          
            'axes.titlesize': 20,    
            'axes.labelsize': 20,      
            'xtick.labelsize': 20,
            'ytick.labelsize': 20,
        })
        
        binary_indices = self.binary_indices
        n_features = len(binary_indices)
        nrows = 3
        ncols = math.ceil(n_features / nrows)
        
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, 8))
        axes = axes.flatten()
        attention_weights = F.softmax(torch.exp(self.model.feature_weights), dim=0)
        for idx, feature_idx in enumerate(binary_indices):
            f1 = self.model.fs[feature_idx].forward(torch.tensor([1.0]).view(-1, 1))\
                     .detach().flatten()[selected_class].item()
            f0 = self.model.fs[feature_idx].forward(torch.tensor([0.0]).view(-1, 1))\
                     .detach().flatten()[selected_class].item()
            weights = float(attention_weights[idx])
            f1 = f1 * weights
            f0 = f0 * weights
            
            ax = axes[idx]
            positions = [0, 0.5]
            bar_width = 0.3
            ax.bar(positions, [f1, f0], color=['#4c72b0', '#dd8452'], width=bar_width, align='center')
            ax.set_xticks(positions)
            ax.set_xticklabels(['$f(1)$', '$f(0)$'])
            ax.axhline(0, color='black', linewidth=1)

            ax.grid(False)
            ax.set_title(self.feature_names[feature_idx])

            max_val = max(abs(f1), abs(f0))
            if max_val == 0:
                max_val = 1
            margin = max_val * 0.1
            ax.set_ylim([-max_val - margin, max_val + margin])

        for j in range(idx + 1, len(axes)):
            fig.delaxes(axes[j])
        
        fig.suptitle(f'Class: {self.class_names[selected_class]}', fontsize=22, fontweight='bold', y=0.85)
        plt.tight_layout(rect=[0, 0, 1, 0.90], h_pad=0.5, w_pad=0.5)
        plt.subplots_adjust(wspace=0.75, hspace=0.5)
        
        self.last_fig = fig
        plt.show()
    
    def save_plot(self, b):
        fname = self.filename_text.value.strip()
        if not fname:
            fname = str(self.data_name) + '_' + str(self.class_names[self.dropdown.value])
            print(fname)
        elif self.last_fig is None:
            print("No plot is available to save.")
        directory = "./plot"
        if not os.path.exists(directory):
            os.makedirs(directory)
        self.last_fig.savefig(f"{directory}/{fname}.pdf", format="pdf", bbox_inches="tight")

        print(f"Plot saved as {fname}.pdf")
    
    def display(self):
        # Use interact to update the plot based on dropdown selection
        widgets.interact(self.plot_f_scores, selected_class=self.dropdown)
        # Display the filename textbox and save button below the plot widget
        display(self.filename_text, self.save_button)

In [ ]:
data_name = 'zoo'
data_path = f'processed_data/{data_name}_smax1.pt'
data = torch.load(data_path, weights_only=False)

model_dict = 'models/zoo_HGNAM_2732_best_val_acc.pt'
model = HGNAN(in_channels=16, hidden_channels=256, num_layers=5, out_channels=7, dropout=0.0, limited_m=True, bias=True, normalize_m=True, weight=True, aggregation='neighbor')
model.load_state_dict(torch.load(f"{model_dict}", map_location=torch.device('cuda'), weights_only=False))

<All keys matched successfully>

In [30]:
class_names = {
    0: 'Mammals', 1: 'Birds', 2: 'Reptiles',
    3: 'Fish', 4: 'Amphibians', 5: 'Insects', 6: 'Invertebrates'
}

plotter = FScorePlotter(model, data.x, data_name, feature_names, class_names)
plotter.display()

interactive(children=(Dropdown(description='Class:', options=(('0: Mammals', 0), ('1: Birds', 1), ('2: Reptile…

Text(value='', description='File Name:', placeholder='Enter file name')

Button(description='Save Plot', icon='save', style=ButtonStyle(), tooltip='Click to save the current plot')